# Danny's Pizza — Dataset Scaffold (OscarF Datasets generator)

> Add blockquote



**Created:** 2025-08-22 00:39

This notebook sets up the **small tables** in-notebook and expects the **large order tables** to be loaded from CSV/XLS files.
It strictly follows the schema from class:

- `pizza_names(pizza_id INT, pizza_name TEXT)`
- `pizza_toppings(topping_id INT, topping_name TEXT)`
- `pizza_recipes(pizza_id INT, toppings TEXT)` where `toppings` is a comma-separated list of `topping_id`s
- `runners(runner_id INT, registration_date DATE)`
- `customer_orders(order_id INT, customer_id INT, pizza_id INT, exclusions VARCHAR(4), extras VARCHAR(4), order_date TIMESTAMP)`
- `runner_orders(order_id INT, runner_id INT, pickup_time VARCHAR(19), distance VARCHAR(7), duration VARCHAR(10), cancellation VARCHAR(23))`



In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')
#base_path = '/content/drive/MyDrive/BLOQUE I.A./Notebooks/'
%ls

In [2]:
import pandas as pd

## Create Small Tables (Dims)

We define a **minimal, realistic** pizza catalog.


In [3]:
# --- pizza_names ---
pizza_names = pd.DataFrame({
    'pizza_id': [1, 2, 3, 4, 5, 6, 7],
    'pizza_name': [
        'Margherita',        # 1
        'Vegetarian',        # 2
        'Meat Lovers',       # 3
        'BBQ Chicken',       # 4
        'Hawaiian',          # 5
        'Pepperoni',         # 6
        'Vegan Veggie'       # 7  <-- vegan option (no cheese by default)
    ]
})
pizza_names.to_csv('pizza_names.csv', index=False)
pizza_names

,pizza_id,pizza_name
0,1,Margherita
1,2,Vegetarian
2,3,Meat Lovers
3,4,BBQ Chicken
4,5,Hawaiian
5,6,Pepperoni
6,7,Vegan Veggie


In [4]:
# --- pizza_toppings ---
# Concise list so students can reason about extras/exclusions clearly
pizza_toppings = pd.DataFrame({
    'topping_id': list(range(1, 16)),
    'topping_name': [
        'Tomato Sauce',  # 1
        'Mozzarella',    # 2
        'Mushroom',      # 3
        'Onion',         # 4
        'Bell Pepper',   # 5
        'Olives',        # 6
        'Pepperoni',     # 7
        'Bacon',         # 8
        'Beef',          # 9
        'Chicken',       # 10
        'Pineapple',     # 11
        'BBQ Sauce',     # 12
        'Jalapeno',      # 13
        'Fresh Basil',   # 14
        'Garlic'         # 15
    ]
})
pizza_toppings.to_csv('pizza_toppings.csv', index=False)
pizza_toppings

,topping_id,topping_name
0,1,Tomato Sauce
1,2,Mozzarella
2,3,Mushroom
3,4,Onion
4,5,Bell Pepper
5,6,Olives
6,7,Pepperoni
7,8,Bacon
8,9,Beef
9,10,Chicken


In [5]:
# --- pizza_recipes ---
# Define base recipes as comma-separated topping_id strings (order does not matter)
recipes_map = {
    1: [1,2,14],               # Margherita: sauce, mozzarella, basil
    2: [1,2,3,4,5,6],          # Vegetarian: sauce, mozzarella, mushroom, onion, bell pepper, olives
    3: [1,2,7,8,9],            # Meat Lovers: sauce, mozzarella, pepperoni, bacon, beef
    4: [1,2,10,12],            # BBQ Chicken: sauce, mozzarella, chicken, bbq sauce
    5: [1,2,11,6],             # Hawaiian: sauce, mozzarella, pineapple, olives
    6: [1,2,7],                # Pepperoni: sauce, mozzarella, pepperoni
    7: [1,3,4,5,6,15]          # Vegan Veggie: sauce, mushroom, onion, bell pepper, olives, garlic (no cheese)
}

pizza_recipes = pd.DataFrame({
    'pizza_id': list(recipes_map.keys()),
    'toppings': [','.join(map(str, v)) for v in recipes_map.values()]
})
pizza_recipes.to_csv('pizza_recipes.csv', index=False)
pizza_recipes

,pizza_id,toppings
0,1,"1,2,14"
1,2,"1,2,3,4,5,6"
2,3,"1,2,7,8,9"
3,4,"1,2,10,12"
4,5,"1,2,11,6"
5,6,"1,2,7"
6,7,"1,3,4,5,6,15"


In [6]:
# --- runners ---

dates = pd.date_range('2021-01-03', periods=15, freq='7D')
runners = pd.DataFrame({
    'runner_id': range(1, 16),
    'registration_date': dates.date
})
runners.to_csv('runners.csv', index=False)
runners

,runner_id,registration_date
0,1,2021-01-03
1,2,2021-01-10
2,3,2021-01-17
3,4,2021-01-24
4,5,2021-01-31
5,6,2021-02-07
6,7,2021-02-14
7,8,2021-02-21
8,9,2021-02-28
9,10,2021-03-07


## Load Orders

This cell loads the orders from CSV first; if not present, it tries XLSX. Adjust file paths if needed.


## Cleaning & Normalization Helpers

- Parse `distance` (to float km) and `duration` (to minutes).
- Normalize `cancellation` labels (lowercase, strip).
- Enforce FK integrity and logical constraints.


In [7]:

import numpy as np

# Base path in Google Drive


# --- Load small dimension tables ---
pizza_names     = pd.read_csv('pizza_names.csv')
pizza_toppings  = pd.read_csv('pizza_toppings.csv')
pizza_recipes   = pd.read_csv('pizza_recipes.csv')
runners         = pd.read_csv('runners.csv')

# --- Load big fact tables ---
customer_orders = pd.read_csv('customer_orders.csv')
runner_orders   = pd.read_csv('runner_orders.csv')

print("pizza_names:", pizza_names.shape)
print("pizza_toppings:", pizza_toppings.shape)
print("pizza_recipes:", pizza_recipes.shape)
print("runners:", runners.shape)
print("customer_orders:", customer_orders.shape)
print("runner_orders:", runner_orders.shape)


pizza_names: (7, 2)
pizza_toppings: (15, 2)
pizza_recipes: (7, 2)
runners: (15, 2)
customer_orders: (2101, 6)
runner_orders: (1500, 6)


In [8]:
customer_orders.head()

,order_id,customer_id,pizza_id,exclusions,extras,order_date
0,2481,500,2,NaN,NaN,2025-02-21 16:06:46
1,1897,269,6,NaN,NaN,2025-08-12 08:27:50
2,1842,59,7,NaN,NaN,2024-12-23 20:50:23
3,1869,710,6,NaN,NaN,2025-01-31 09:16:49
4,2384,189,3,NaN,NaN,2024-11-30 17:35:37




```
# This is formatted as code
```


# /* --------------------
#   Case Study Questions
#   --------------------*/
A. Pizza Metrics

    How many pizzas were ordered?
    How many unique customer orders were made?
    How many successful orders were delivered by each runner?
    How many of each type of pizza was delivered?
    How many Vegetarian and Meatlovers were ordered by each customer?
    What was the maximum number of pizzas delivered in a single order?
    For each customer, how many delivered pizzas had at least 1 change and how many had no changes?
    How many pizzas were delivered that had both exclusions and extras?
    What was the total volume of pizzas ordered for each hour of the day?
    What was the volume of orders for each day of the week?

B. Runner and Customer Experience

    How many runners signed up for each 1 week period? (i.e. week starts 2021-01-01)
    What was the average time in minutes it took for each runner to arrive at the Pizza Runner HQ to pickup the order?
    Is there any relationship between the number of pizzas and how long the order takes to prepare?
    What was the average distance travelled for each customer?
    What was the difference between the longest and shortest delivery times for all orders?
    What was the average speed for each runner for each delivery and do you notice any trend for these values?
    What is the successful delivery percentage for each runner?

🍕 section c — customer & business intelligence

C1. total customer spend
 which customers bring in the most revenue?

use to argue for vip memberships or spend-based rewards.

C2. customer frequency (distinct days of orders)
 who orders regularly vs. one-off customers?

segment into loyal customers vs. occasional customers.

hint at frequency-based discounts (e.g., 5th order free).

C3. first pizza ordered by each customer
 what attracts customers initially?

good to identify entry-point pizzas (the hook item that brings people in).

hint at discounts on first-order pizzas to acquire new customers.

C4. overall best-seller pizza
 which pizza keeps the lights on?

highlight as a flagship product to promote.

use for seasonal bundles (“summer deal with our #1 pizza”).

C5. most popular pizza by customer
 can we personalize offers?

recommend personalized “customer favorites” discounts.

hint toward AI/BI-driven recommender systems.

C6. regulars with ≥30 orders and their go-to pizzas
 who are the heavy hitters and what do they like?

obvious loyalty program candidates.

pitch: “keep them happy with exclusive rewards so they don’t churn.”

C7. customers with very consistent habits (always order the same pizza)
 creatures of habit = stable recurring revenue.

membership idea: “pizza subscription” (weekly plan with their pizza auto-delivered).

C8. the “perfect pair”

great marketing story: “find your pizza soulmate.”

pitch: social media campaign + 2-for-1  perfect pizza couples’ promo.

C9. peak order times
👉 what hours & days matter most?

operational: staff scheduling.

marketing: happy hour discounts in slow periods, premium pricing at peak times.

C10. best candidates for loyalty program
👉 combine spend + frequency + consistency.

identify top 5–10% customers.

suggest tiered memberships: silver/gold/platinum.

seasonal perks: double points in winter when sales slow.

# Entregable

final presentation = a business intelligence pitch deck:

customer segmentation (loyal vs occasional vs perfect pair).

menu insights (flagship pizza, first-order hook, personal favorites).

time insights (peak hours, seasonal discounts).

strategic recommendations:

loyalty program design,

subscription/membership tiers,

seasonal & time-based promos,

“perfect pair” marketing campaign.

In [9]:
import sqlite3

db_path = f"dannys_pizza.sqlite"
conn = sqlite3.connect(db_path)
c = conn.cursor()

# drop existing tables (clean slate)
for t in [
    'pizza_names','pizza_toppings','pizza_recipes',
    'runners','customer_orders','runner_orders'
]:
    c.execute(f"DROP TABLE IF EXISTS {t};")

# create empty tables with the canonical column names
c.execute("""CREATE TABLE pizza_names (
  pizza_id INTEGER,
  pizza_name TEXT
);""")

c.execute("""CREATE TABLE pizza_toppings (
  topping_id INTEGER,
  topping_name TEXT
);""")

c.execute("""CREATE TABLE pizza_recipes (
  pizza_id INTEGER,
  toppings TEXT
);""")

c.execute("""CREATE TABLE runners (
  runner_id INTEGER,
  registration_date TEXT
);""")

# NOTE: keep your current column names exactly as they are in the DataFrame
# If your DF uses 'order_date', keep it; if it's 'order_time', keep that.
# Below uses 'order_date'—change to 'order_time' if that’s your DF.
c.execute("""CREATE TABLE customer_orders (
  order_id INTEGER,
  customer_id INTEGER,
  pizza_id INTEGER,
  exclusions TEXT,
  extras TEXT,
  order_date TEXT
);""")

c.execute("""CREATE TABLE runner_orders (
  order_id INTEGER,
  runner_id INTEGER,
  pickup_time TEXT,
  distance TEXT,
  duration TEXT,
  cancellation TEXT
);""")

conn.commit()

# append DataFrames exactly as-is (no cleaning)
pizza_names.to_sql('pizza_names', conn, if_exists='append', index=False)
pizza_toppings.to_sql('pizza_toppings', conn, if_exists='append', index=False)
pizza_recipes.to_sql('pizza_recipes', conn, if_exists='append', index=False)
runners.to_sql('runners', conn, if_exists='append', index=False)
customer_orders.to_sql('customer_orders', conn, if_exists='append', index=False)
runner_orders.to_sql('runner_orders', conn, if_exists='append', index=False)

conn.commit()
print("SQLite ready at:", db_path)


SQLite ready at: dannys_pizza.sqlite


* Q1. How many pizzas were ordered?

In [10]:
pd.read_sql("""
SELECT COUNT(*) AS total_pizzas
FROM customer_orders;
""", conn)


,total_pizzas
0,2101


In [11]:
pd.read_sql("""
SELECT COUNT(DISTINCT order_id) AS unique_orders
FROM customer_orders;
""", conn)


,unique_orders
0,1500


In [12]:
##pd.read_sql("YOUR QUERY HERE", conn)  # TODO


In [13]:
pd.read_sql("""
SELECT runner_id,
       COUNT(runner_id) AS order_count
FROM runner_orders
WHERE distance IS NOT NULL
  AND TRIM(distance) <> ''
  AND cancellation = ''
GROUP BY runner_id
ORDER BY order_count DESC;
""", conn)


,runner_id,order_count


Notice how our query returned empty results? That is a clue something is off in our filter. We wrote
WHERE distance IS NOT NULL
  AND cancellation = ''''

  but in this dataset, the cancellation column does not only use a blank string to mean no cancellation. Sometimes it has the literal word 'null', sometimes it is NULL (the SQL null value), sometimes different casing (Null, NULL). Because of that, our =  condition excluded almost everything.

In [14]:
  pd.read_sql("""
SELECT DISTINCT TRIM(cancellation) AS cancellation_value,
       COUNT(*) AS n
FROM runner_orders
GROUP BY 1
ORDER BY n DESC;
""", conn)


,cancellation_value,n
0,None,1450
1,address issue,13
2,customer no show,9
3,runner sick,7
4,order late,7
5,custmer_no_sho,7
6,runner_unavailable,5
7,restaurant_cancelled,2


In [15]:
# 3a) Successful orders delivered by each runner (ignore distance presence)
pd.read_sql("""
SELECT runner_id,
       COUNT(*) AS order_count
FROM runner_orders
WHERE COALESCE(TRIM(LOWER(cancellation)),'') IN ('', 'null')
GROUP BY runner_id
ORDER BY order_count DESC;
""", conn)


,runner_id,order_count
0,3,115
1,14,112
2,2,110
3,5,109
4,7,101
5,12,96
6,11,96
7,1,93
8,8,92
9,13,91


In [16]:
#How many unique customer orders were made? YAAA 
#How many successful orders were delivered by each runner? YAAA 
#How many of each type of pizza was delivered? YAAA
#How many Vegetarian and Meatlovers were ordered by each customer?
#What was the maximum number of pizzas delivered in a single order?
#For each customer, how many delivered pizzas had at least 1 change and how many had no changes?
#How many pizzas were delivered that had both exclusions and extras?
#What was the total volume of pizzas ordered for each hour of the day?
#What was the volume of orders for each day of the week?

In [17]:
#How many of each type of pizza was delivered?
pd.read_sql("""
SELECT co.pizza_id,pn.pizza_name,COUNT(order_id) 
FROM customer_orders co
JOIN pizza_names as pn
ON co.pizza_id = pn.pizza_id
GROUP BY co.pizza_id
""", conn)


,pizza_id,pizza_name,COUNT(order_id)
0,1,Margherita,235
1,2,Vegetarian,340
2,3,Meat Lovers,468
3,4,BBQ Chicken,278
4,5,Hawaiian,218
5,6,Pepperoni,398
6,7,Vegan Veggie,164


In [18]:
#How many Vegetarian and Meatlovers were ordered by each customer?
pd.read_sql("""
SELECT co.customer_id,COUNT(order_id) as ordenes 
FROM customer_orders co
JOIN pizza_names as pn
ON co.pizza_id = pn.pizza_id
WHERE co.pizza_id IN (2,3)
GROUP BY co.pizza_id, co.customer_id
""", conn)

,customer_id,ordenes
0,5,4
1,11,1
2,15,1
3,16,1
4,17,1
...,...,...
433,778,1
434,781,1
435,785,1
436,790,1


In [19]:
#What was the maximum number of pizzas delivered in a single order?
pd.read_sql("""
SELECT order_id, COUNT(pizza_id)
FROM customer_orders
GROUP BY order_id
ORDER BY COUNT(pizza_id) DESC
""", conn)

,order_id,COUNT(pizza_id)
0,2441,4
1,2417,4
2,1994,4
3,1930,4
4,1805,4
...,...,...
1495,1007,1
1496,1005,1
1497,1003,1
1498,1002,1


In [20]:
#How many pizzas were delivered that had both exclusions and extras?
pd.read_sql("""
SELECT COUNT(*)
FROM customer_orders
WHERE extras IS NOT NULL and exclusions IS NOT NULL""", conn)

,COUNT(*)
0,198


In [21]:
#What was the total volume of pizzas ordered for each hour of the day?
pd.read_sql("""
SELECT COUNT(order_id),strftime('%H', order_date) as hora
FROM customer_orders
GROUP BY strftime('%H', order_date)""", conn)


,COUNT(order_id),hora
0,66,00
1,39,01
2,49,02
3,59,03
4,54,04
5,28,05
6,74,06
7,36,07
8,43,08
9,67,09


In [22]:
#What was the volume of orders for each day of the week?
pd.read_sql("""
SELECT 
    CASE CAST( strftime('%w', order_date) as INTEGER)
    WHEN 0 THEN 'Sunday'
    WHEN 1 THEN 'Monday'
    WHEN 2 THEN 'Tuesday'
    WHEN 3 THEN 'Wednesday'
    WHEN 4 THEN 'Thursday'
    WHEN 5 THEN 'Friday'
    ELSE 'Saturday'
    END AS dia,
    COUNT(order_id)
FROM customer_orders
GROUP BY strftime('%w', order_date)""", conn)

,dia,COUNT(order_id)
0,Sunday,439
1,Monday,176
2,Tuesday,177
3,Wednesday,190
4,Thursday,251
5,Friday,417
6,Saturday,451


In [23]:
#

In [24]:
pd.read_sql("""
SELECT co.customer_id,COUNT(order_id) as ordenes 
FROM customer_orders co
GROUP BY co.customer_id
ORDER BY ordenes DESC
LIMIT 18""", conn)

,customer_id,ordenes
0,620,58
1,500,54
2,730,53
3,657,52
4,30,52
5,717,51
6,547,49
7,180,49
8,240,47
9,45,45


In [25]:
#What was the month 
pd.read_sql("""
SELECT 
    CASE CAST( strftime('%m', order_date) as INTEGER)
    WHEN 1 THEN 'Enero'
    WHEN 2 THEN 'Febrero'
    WHEN 3 THEN 'Marzo'
    WHEN 4 THEN 'Abril'
    WHEN 5 THEN 'Mayo'
    WHEN 6 THEN 'Junio'
    WHEN 7 THEN 'Julio'
    WHEN 8 THEN 'Agosto'
    WHEN 9 THEN 'Septiembre'
    WHEN 10 THEN 'Octubre'
    WHEN 11 THEN 'Noviembre'
    WHEN 12 THEN 'Diciembre'
    ELSE 'Mes inválido'
    END AS dia,
    COUNT(order_id)
FROM customer_orders
GROUP BY strftime('%m', order_date)""", conn)

,dia,COUNT(order_id)
0,Enero,238
1,Febrero,244
2,Marzo,202
3,Abril,184
4,Mayo,254
5,Junio,230
6,Julio,208
7,Agosto,108
8,Noviembre,226
9,Diciembre,207


In [26]:
#What was the month 
new_df = pd.read_sql("""
SELECT customer_id, pizza_id,sum(pizza_id) as cantidad
FROM customer_orders
GROUP BY customer_id, pizza_id
""", conn)

In [27]:
pivot_df = pd.pivot_table(new_df, values='cantidad', index=['customer_id'], columns=['pizza_id'], aggfunc="sum")

In [28]:
pivot_df.head(14)

pizza_id,1,2,3,4,5,6,7
customer_id,,,,,,,
1,NaN,NaN,3.0,4.0,NaN,6.0,NaN
2,1.0,NaN,3.0,NaN,NaN,NaN,NaN
3,NaN,NaN,3.0,NaN,NaN,6.0,NaN
4,1.0,NaN,NaN,NaN,NaN,NaN,NaN
5,2.0,8.0,21.0,28.0,15.0,36.0,21.0
7,NaN,NaN,NaN,4.0,NaN,NaN,NaN
8,1.0,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,6.0,7.0
10,NaN,NaN,3.0,NaN,NaN,NaN,NaN


In [29]:
pivot_df = pivot_df.fillna(0)

In [30]:
pivot_df.head(14)

pizza_id,1,2,3,4,5,6,7
customer_id,,,,,,,
1,0.0,0.0,3.0,4.0,0.0,6.0,0.0
2,1.0,0.0,3.0,0.0,0.0,0.0,0.0
3,0.0,0.0,3.0,0.0,0.0,6.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0
5,2.0,8.0,21.0,28.0,15.0,36.0,21.0
7,0.0,0.0,0.0,4.0,0.0,0.0,0.0
8,1.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,6.0,7.0
10,0.0,0.0,3.0,0.0,0.0,0.0,0.0


In [31]:
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(pivot_df)
import pandas as pd
sim_df = pd.DataFrame(sim_matrix, index=pivot_df.index, columns=pivot_df.index)
print(sim_df.head())


customer_id       1         2         3         4         5         7    \
customer_id                                                               
1            1.000000  0.364399  0.858898  0.000000  0.877478  0.512148   
2            0.364399  1.000000  0.424264  0.316228  0.360278  0.000000   
3            0.858898  0.424264  1.000000  0.000000  0.728991  0.000000   
4            0.000000  0.316228  0.000000  1.000000  0.035055  0.000000   
5            0.877478  0.360278  0.728991  0.035055  1.000000  0.490775   

customer_id       8         9         10        11   ...       782       783  \
customer_id                                          ...                       
1            0.000000  0.499952  0.384111  0.000000  ...  0.765568  0.000000   
2            0.316228  0.000000  0.948683  0.000000  ...  0.026261  0.000000   
3            0.000000  0.582086  0.447214  0.000000  ...  0.891338  0.000000   
4            1.000000  0.000000  0.000000  0.000000  ...  0.083045  0.0000

In [32]:
# Para cada cliente, encontrar el gemelo más parecido
gemelos = {}

for customer in sim_df.index:
    # Excluirse a sí mismo
    similitudes = sim_df.loc[customer].drop(customer)
    gemelo = similitudes.idxmax()
    score = similitudes.max()
    gemelos[customer] = (gemelo, score)

# Ver resultados
for cliente, (gemelo, score) in gemelos.items():
    if score==1.0:
        print(f"Cliente {cliente} es más parecido a Cliente {gemelo} (similitud={score:.2f})")

Cliente 1 es más parecido a Cliente 14 (similitud=1.00)
Cliente 2 es más parecido a Cliente 122 (similitud=1.00)
Cliente 4 es más parecido a Cliente 8 (similitud=1.00)
Cliente 7 es más parecido a Cliente 60 (similitud=1.00)
Cliente 8 es más parecido a Cliente 4 (similitud=1.00)
Cliente 9 es más parecido a Cliente 28 (similitud=1.00)
Cliente 10 es más parecido a Cliente 24 (similitud=1.00)
Cliente 11 es más parecido a Cliente 64 (similitud=1.00)
Cliente 12 es más parecido a Cliente 383 (similitud=1.00)
Cliente 14 es más parecido a Cliente 1 (similitud=1.00)
Cliente 15 es más parecido a Cliente 266 (similitud=1.00)
Cliente 16 es más parecido a Cliente 459 (similitud=1.00)
Cliente 22 es más parecido a Cliente 33 (similitud=1.00)
Cliente 24 es más parecido a Cliente 10 (similitud=1.00)
Cliente 27 es más parecido a Cliente 10 (similitud=1.00)
Cliente 28 es más parecido a Cliente 9 (similitud=1.00)
Cliente 33 es más parecido a Cliente 22 (similitud=1.00)
Cliente 35 es más parecido a Cliente 

In [33]:
#What was the month 
new_df2 = pd.read_sql("""
SELECT customer_id, order_id, pizza_id,sum(pizza_id) as cantidad
FROM customer_orders
GROUP BY customer_id, pizza_id
""", conn)

In [64]:
new_df2

,customer_id,order_id,pizza_id,cantidad
0,1,2460,3,3
1,1,2460,4,4
2,1,1990,6,6
3,2,2444,1,1
4,2,2444,3,3
...,...,...,...,...
1218,794,1840,6,12
1219,796,1480,1,1
1220,796,2491,2,4
1221,796,1837,6,6


In [65]:
pivot_df2 = pd.pivot_table(new_df2, values='cantidad', index=['customer_id','order_id'], columns=['pizza_id'], aggfunc="sum")

In [66]:
pivot_df2.head()

pizza_id                1   2    3    4   5    6   7
customer_id order_id                                
1           1990      NaN NaN  NaN  NaN NaN  6.0 NaN
            2460      NaN NaN  3.0  4.0 NaN  NaN NaN
2           2444      1.0 NaN  3.0  NaN NaN  NaN NaN
3           1197      NaN NaN  3.0  NaN NaN  NaN NaN
            1989      NaN NaN  NaN  NaN NaN  6.0 NaN

In [67]:
pivot_df2 = pivot_df2.fillna(0)

In [68]:
pivot_df2.head()

pizza_id                1    2    3    4    5    6    7
customer_id order_id                                   
1           1990      0.0  0.0  0.0  0.0  0.0  6.0  0.0
            2460      0.0  0.0  3.0  4.0  0.0  0.0  0.0
2           2444      1.0  0.0  3.0  0.0  0.0  0.0  0.0
3           1197      0.0  0.0  3.0  0.0  0.0  0.0  0.0
            1989      0.0  0.0  0.0  0.0  0.0  6.0  0.0

In [39]:
corr_pedidos = pivot_df2.T.corr(method='pearson')
print(corr_pedidos)

customer_id                1                   2         3              \
order_id                  1990      2460      2444      1197      1989   
customer_id order_id                                                     
1           1990      1.000000 -0.254588 -0.222222 -0.166667  1.000000   
            2460     -0.254588  1.000000  0.424313  0.509175 -0.254588   
2           2444     -0.222222  0.424313  1.000000  0.944444 -0.222222   
3           1197     -0.166667  0.509175  0.944444  1.000000 -0.166667   
            1989      1.000000 -0.254588 -0.222222 -0.166667  1.000000   
...                        ...       ...       ...       ...       ...   
794         1840      1.000000 -0.254588 -0.222222 -0.166667  1.000000   
796         1480     -0.166667 -0.254588  0.166667 -0.166667 -0.166667   
            1837      1.000000 -0.254588 -0.222222 -0.166667  1.000000   
            2491     -0.166667 -0.254588 -0.222222 -0.166667 -0.166667   
798         1286     -0.166667 -0.2545

In [40]:
corr_pedidos.head()

customer_id                1                   2         3              \
order_id                  1990      2460      2444      1197      1989   
customer_id order_id                                                     
1           1990      1.000000 -0.254588 -0.222222 -0.166667  1.000000   
            2460     -0.254588  1.000000  0.424313  0.509175 -0.254588   
2           2444     -0.222222  0.424313  1.000000  0.944444 -0.222222   
3           1197     -0.166667  0.509175  0.944444  1.000000 -0.166667   
            1989      1.000000 -0.254588 -0.222222 -0.166667  1.000000   

customer_id                4         5                                  ...  \
order_id                  1294      1131      1289      1398      1804  ...   
customer_id order_id                                                    ...   
1           1990     -0.166667 -0.166667 -0.166667 -0.180266 -0.166667  ...   
            2460     -0.254588 -0.254588 -0.254588  0.752651 -0.254588  ...   
2           2444      0.166667 -0.222222 -0.222222 -0.212313 -0.222222  ...   
3           1197     -0.166667 -0.166667 -0.166667 -0.180266 -0.166667  ...   
            1989     -0.166667 -0.166667 -0.166667 -0.180266 -0.166667  ...   

customer_id                790                           791       793  \
order_id                  1399      1772      2087      1612      1064   
customer_id order_id                                                     
1           1990     -0.166667 -0.166667 -0.166667 -0.166667 -0.166667   
            2460     -0.254588  0.509175 -0.254588 -0.254588  0.509175   
2           2444      0.166667  0.944444 -0.222222 -0.222222  0.944444   
3           1197     -0.166667  1.000000 -0.166667 -0.166667  1.000000   
            1989     -0.166667 -0.166667 -0.166667 -0.166667 -0.166667   

customer_id                794       796                           798  
order_id                  1840      1480      1837      2491      1286  
customer_id order_id                                                    
1           1990      1.000000 -0.166667  1.000000 -0.166667 -0.166667  
            2460     -0.254588 -0.254588 -0.254588 -0.254588 -0.254588  
2           2444     -0.222222  0.166667 -0.222222 -0.222222 -0.222222  
3           1197     -0.166667 -0.166667 -0.166667 -0.166667 -0.166667  
            1989      1.000000 -0.166667  1.000000 -0.166667 -0.166667  

[5 rows x 995 columns]

In [46]:
# Información básica de la matriz
print(f"Forma de la matriz: {corr_pedidos.shape}")
print(f"Rango de correlaciones: {corr_pedidos.min().min():.3f} a {corr_pedidos.max().max():.3f}")

# Estadísticas descriptivas (excluyendo la diagonal)
import numpy as np
mask = np.eye(corr_pedidos.shape[0], dtype=bool)
corr_sin_diagonal = corr_pedidos.values[~mask]
print(f"Media de correlaciones: {np.mean(corr_sin_diagonal):.3f}")
print(f"Desviación estándar: {np.std(corr_sin_diagonal):.3f}")

Forma de la matriz: (995, 995)
Rango de correlaciones: -0.602 a 1.000
Media de correlaciones: 0.014
Desviación estándar: 0.418


In [47]:
# Método alternativo más claro
import numpy as np

# Obtener solo la triangular inferior (sin diagonal)
indices = np.tril_indices_from(corr_pedidos, k=-1)
valores_correlacion = corr_pedidos.values[indices]
indices_nombres = [(corr_pedidos.index[i], corr_pedidos.columns[j]) 
                   for i, j in zip(indices[0], indices[1])]

# Crear Serie con los pares de nombres como índice
correlaciones_serie = pd.Series(valores_correlacion, 
                                index=pd.MultiIndex.from_tuples(indices_nombres))

# Top correlaciones
print("Top 10 correlaciones positivas:")
print(correlaciones_serie.nlargest(10))

print("\nTop 10 correlaciones negativas:")
print(correlaciones_serie.nsmallest(10))

Top 10 correlaciones positivas:
(17, 2379)  (5, 1289)     1.0
(30, 1219)  (5, 1131)     1.0
            (11, 1460)    1.0
            (15, 1495)    1.0
            (18, 1230)    1.0
            (29, 1943)    1.0
(35, 2273)  (30, 1219)    1.0
(39, 1764)  (5, 1289)     1.0
(45, 1160)  (17, 2379)    1.0
            (39, 1764)    1.0
dtype: float64

Top 10 correlaciones negativas:
(378, 1778)  (159, 1923)   -0.602026
             (274, 2255)   -0.599008
(745, 2237)  (672, 2070)   -0.589564
(605, 1994)  (525, 2450)   -0.585226
(745, 2237)  (444, 1299)   -0.574176
             (455, 1271)   -0.574176
(605, 1994)  (208, 2351)   -0.569047
             (414, 1159)   -0.569047
             (130, 1519)   -0.563099
             (402, 1303)   -0.563099
dtype: float64


In [48]:
correlaciones_serie.nlargest(50)

(17, 2379)   (5, 1289)     1.0
(30, 1219)   (5, 1131)     1.0
             (11, 1460)    1.0
             (15, 1495)    1.0
             (18, 1230)    1.0
             (29, 1943)    1.0
(35, 2273)   (30, 1219)    1.0
(39, 1764)   (5, 1289)     1.0
(45, 1160)   (17, 2379)    1.0
             (39, 1764)    1.0
(49, 1352)   (5, 1289)     1.0
             (45, 1160)    1.0
(62, 1444)   (30, 1219)    1.0
(64, 1707)   (30, 1219)    1.0
(73, 1009)   (30, 1219)    1.0
(74, 1472)   (5, 1289)     1.0
             (45, 1160)    1.0
(80, 1661)   (30, 1219)    1.0
(81, 2121)   (30, 1219)    1.0
(82, 1579)   (30, 1219)    1.0
(85, 1104)   (30, 1219)    1.0
(95, 2492)   (5, 1289)     1.0
             (45, 1160)    1.0
(96, 1641)   (5, 1289)     1.0
             (45, 1160)    1.0
(97, 1859)   (30, 1219)    1.0
(99, 2024)   (5, 1289)     1.0
             (45, 1160)    1.0
(103, 1985)  (30, 1219)    1.0
(105, 1695)  (5, 1289)     1.0
             (45, 1160)    1.0
(107, 2360)  (30, 1219)    1.0
(108, 23

In [49]:
correlaciones_serie

(1, 2460)    (1, 1990)     -0.254588
(2, 2444)    (1, 1990)     -0.222222
             (1, 2460)      0.424313
(3, 1197)    (1, 1990)     -0.166667
             (1, 2460)      0.509175
                              ...   
(798, 1286)  (793, 1064)   -0.166667
             (794, 1840)   -0.166667
             (796, 1480)   -0.166667
             (796, 1837)   -0.166667
             (796, 2491)    1.000000
Length: 494515, dtype: float64

In [50]:
count

NameError: name 'count' is not defined

In [51]:
corr_pedidos.loc[5]

customer_id       1                   2         3                   4    \
order_id         1990      2460      2444      1197      1989      1294   
order_id                                                                  
1131        -0.166667 -0.254588 -0.222222 -0.166667 -0.166667 -0.166667   
1289        -0.166667 -0.254588 -0.222222 -0.166667 -0.166667 -0.166667   
1398        -0.180266  0.752651 -0.212313 -0.180266 -0.180266 -0.096142   
1804        -0.166667 -0.254588 -0.222222 -0.166667 -0.166667 -0.166667   
2091         1.000000 -0.254588 -0.222222 -0.166667  1.000000 -0.166667   
2404        -0.166667  0.509175  0.944444  1.000000 -0.166667 -0.166667   

customer_id       5                                  ...       790            \
order_id         1131      1289      1398      1804  ...      1399      1772   
order_id                                             ...                       
1131         1.000000 -0.166667 -0.180266 -0.166667  ... -0.166667 -0.166667   
1289        -0.166667  1.000000 -0.180266 -0.166667  ... -0.166667 -0.166667   
1398        -0.180266 -0.180266  1.000000 -0.180266  ... -0.096142 -0.180266   
1804        -0.166667 -0.166667 -0.180266  1.000000  ... -0.166667 -0.166667   
2091        -0.166667 -0.166667 -0.180266 -0.166667  ... -0.166667 -0.166667   
2404        -0.166667 -0.166667 -0.180266 -0.166667  ... -0.166667  1.000000   

customer_id                 791       793       794       796            \
order_id         2087      1612      1064      1840      1480      1837   
order_id                                                                  
1131         1.000000  1.000000 -0.166667 -0.166667 -0.166667 -0.166667   
1289        -0.166667 -0.166667 -0.166667 -0.166667 -0.166667 -0.166667   
1398        -0.180266 -0.180266 -0.180266 -0.180266 -0.096142 -0.180266   
1804        -0.166667 -0.166667 -0.166667 -0.166667 -0.166667 -0.166667   
2091        -0.166667 -0.166667 -0.166667  1.000000 -0.166667  1.000000   
2404        -0.166667 -0.166667  1.000000 -0.166667 -0.166667 -0.166667   

customer_id                 798  
order_id         2491      1286  
order_id                         
1131         1.000000  1.000000  
1289        -0.166667 -0.166667  
1398        -0.180266 -0.180266  
1804        -0.166667 -0.166667  
2091        -0.166667 -0.166667  
2404        -0.166667 -0.166667  

[6 rows x 995 columns]

In [57]:
pivot_df2.head(20)

pizza_id                1    2     3     4     5     6     7
customer_id order_id                                        
1           1990      0.0  0.0   0.0   0.0   0.0   6.0   0.0
            2460      0.0  0.0   3.0   4.0   0.0   0.0   0.0
2           2444      1.0  0.0   3.0   0.0   0.0   0.0   0.0
3           1197      0.0  0.0   3.0   0.0   0.0   0.0   0.0
            1989      0.0  0.0   0.0   0.0   0.0   6.0   0.0
4           1294      1.0  0.0   0.0   0.0   0.0   0.0   0.0
5           1131      0.0  8.0   0.0   0.0   0.0   0.0   0.0
            1289      0.0  0.0   0.0   0.0  15.0   0.0   0.0
            1398      2.0  0.0   0.0  28.0   0.0   0.0   0.0
            1804      0.0  0.0   0.0   0.0   0.0   0.0  21.0
            2091      0.0  0.0   0.0   0.0   0.0  36.0   0.0
            2404      0.0  0.0  21.0   0.0   0.0   0.0   0.0
7           1518      0.0  0.0   0.0   4.0   0.0   0.0   0.0
8           1326      1.0  0.0   0.0   0.0   0.0   0.0   0.0
9           1251      0.0  0.0   0.0   0.0   0.0   6.0   7.0
10          2014      0.0  0.0   3.0   0.0   0.0   0.0   0.0
11          1460      0.0  2.0   0.0   0.0   0.0   0.0   0.0
12          1322      0.0  0.0   0.0   0.0   5.0   6.0   0.0
13          1263      0.0  0.0   6.0   0.0   0.0   6.0   0.0
14          1192      0.0  0.0   3.0   0.0   0.0   0.0   0.0

In [55]:
pivot_df2.loc[(5,:)]


SyntaxError: invalid syntax (1653017736.py, line 1)

In [58]:
pivot_df2.loc[(5, 1131)]


pizza_id
1    0.0
2    8.0
3    0.0
4    0.0
5    0.0
6    0.0
7    0.0
Name: (5, 1131), dtype: float64

In [59]:
pivot_df2.loc[5]


pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1131,0.0,8.0,0.0,0.0,0.0,0.0,0.0
1289,0.0,0.0,0.0,0.0,15.0,0.0,0.0
1398,2.0,0.0,0.0,28.0,0.0,0.0,0.0
1804,0.0,0.0,0.0,0.0,0.0,0.0,21.0
2091,0.0,0.0,0.0,0.0,0.0,36.0,0.0
2404,0.0,0.0,21.0,0.0,0.0,0.0,0.0


In [60]:
pivot_df2.loc[45]



pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1160,0.0,0.0,0.0,0.0,15.0,0.0,0.0
1388,0.0,14.0,0.0,0.0,0.0,0.0,0.0
1490,0.0,0.0,0.0,36.0,0.0,0.0,0.0
1871,0.0,0.0,21.0,0.0,0.0,0.0,0.0
1895,0.0,0.0,0.0,0.0,0.0,0.0,14.0
1952,0.0,0.0,0.0,0.0,0.0,66.0,0.0
1957,6.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
pivot_df2.head()

pizza_id                1    2    3    4    5    6    7
customer_id order_id                                   
1           1990      0.0  0.0  0.0  0.0  0.0  6.0  0.0
            2460      0.0  0.0  3.0  4.0  0.0  0.0  0.0
2           2444      1.0  0.0  3.0  0.0  0.0  0.0  0.0
3           1197      0.0  0.0  3.0  0.0  0.0  0.0  0.0
            1989      0.0  0.0  0.0  0.0  0.0  6.0  0.0

In [70]:
user_corr = pivot_df2.T.corr()

In [71]:
user_corr

customer_id                1                   2         3              \
order_id                  1990      2460      2444      1197      1989   
customer_id order_id                                                     
1           1990      1.000000 -0.254588 -0.222222 -0.166667  1.000000   
            2460     -0.254588  1.000000  0.424313  0.509175 -0.254588   
2           2444     -0.222222  0.424313  1.000000  0.944444 -0.222222   
3           1197     -0.166667  0.509175  0.944444  1.000000 -0.166667   
            1989      1.000000 -0.254588 -0.222222 -0.166667  1.000000   
...                        ...       ...       ...       ...       ...   
794         1840      1.000000 -0.254588 -0.222222 -0.166667  1.000000   
796         1480     -0.166667 -0.254588  0.166667 -0.166667 -0.166667   
            1837      1.000000 -0.254588 -0.222222 -0.166667  1.000000   
            2491     -0.166667 -0.254588 -0.222222 -0.166667 -0.166667   
798         1286     -0.166667 -0.254588 -0.222222 -0.166667 -0.166667   

customer_id                4         5                                  ...  \
order_id                  1294      1131      1289      1398      1804  ...   
customer_id order_id                                                    ...   
1           1990     -0.166667 -0.166667 -0.166667 -0.180266 -0.166667  ...   
            2460     -0.254588 -0.254588 -0.254588  0.752651 -0.254588  ...   
2           2444      0.166667 -0.222222 -0.222222 -0.212313 -0.222222  ...   
3           1197     -0.166667 -0.166667 -0.166667 -0.180266 -0.166667  ...   
            1989     -0.166667 -0.166667 -0.166667 -0.180266 -0.166667  ...   
...                        ...       ...       ...       ...       ...  ...   
794         1840     -0.166667 -0.166667 -0.166667 -0.180266 -0.166667  ...   
796         1480      1.000000 -0.166667 -0.166667 -0.096142 -0.166667  ...   
            1837     -0.166667 -0.166667 -0.166667 -0.180266 -0.166667  ...   
            2491     -0.166667  1.000000 -0.166667 -0.180266 -0.166667  ...   
798         1286     -0.166667  1.000000 -0.166667 -0.180266 -0.166667  ...   

customer_id                790                           791       793  \
order_id                  1399      1772      2087      1612      1064   
customer_id order_id                                                     
1           1990     -0.166667 -0.166667 -0.166667 -0.166667 -0.166667   
            2460     -0.254588  0.509175 -0.254588 -0.254588  0.509175   
2           2444      0.166667  0.944444 -0.222222 -0.222222  0.944444   
3           1197     -0.166667  1.000000 -0.166667 -0.166667  1.000000   
            1989     -0.166667 -0.166667 -0.166667 -0.166667 -0.166667   
...                        ...       ...       ...       ...       ...   
794         1840     -0.166667 -0.166667 -0.166667 -0.166667 -0.166667   
796         1480      1.000000 -0.166667 -0.166667 -0.166667 -0.166667   
            1837     -0.166667 -0.166667 -0.166667 -0.166667 -0.166667   
            2491     -0.166667 -0.166667  1.000000  1.000000 -0.166667   
798         1286     -0.166667 -0.166667  1.000000  1.000000 -0.166667   

customer_id                794       796                           798  
order_id                  1840      1480      1837      2491      1286  
customer_id order_id                                                    
1           1990      1.000000 -0.166667  1.000000 -0.166667 -0.166667  
            2460     -0.254588 -0.254588 -0.254588 -0.254588 -0.254588  
2           2444     -0.222222  0.166667 -0.222222 -0.222222 -0.222222  
3           1197     -0.166667 -0.166667 -0.166667 -0.166667 -0.166667  
            1989      1.000000 -0.166667  1.000000 -0.166667 -0.166667  
...                        ...       ...       ...       ...       ...  
794         1840      1.000000 -0.166667  1.000000 -0.166667 -0.166667  
796         1480     -0.166667  1.000000 -0.166667 -0.166667 -0.166667  
            1837

In [ ]:
def get_customer_correlations(corr_series, customer_id):
    """
    Obtiene todas las correlaciones de un customer_id específico
    """
    # Filtrar donde el customer_id aparece en cualquiera de los dos niveles
    mask = (
        (corr_series.index.get_level_values(0).str[0] == customer_id) |
        (corr_series.index.get_level_values(1).str[0] == customer_id)
    )
    
    return corr_series[mask].sort_values(ascending=False)

# Usar la función
customer_17_corr = get_customer_correlations(correlaciones_serie, 17)
print(f"Correlaciones del customer 17:")
print(customer_17_corr)

In [111]:
def get_correlation_between_customers(corr_series, customer_id_1, customer_id_2):
    """
    Encuentra todas las correlaciones entre dos customers específicos
    """
    # Filtrar donde ambos customers aparecen en el par
    mask = (
        ((corr_series.index.get_level_values(0).str[0] == customer_id_1) & 
         (corr_series.index.get_level_values(1).str[0] == customer_id_2)) |
        ((corr_series.index.get_level_values(0).str[0] == customer_id_2) & 
         (corr_series.index.get_level_values(1).str[0] == customer_id_1))
    )
    
    return corr_series[mask].sort_values(ascending=False)

# Ejemplo: correlaciones entre customer 698 y 17
corr_698_17 = get_correlation_between_customers(corr_series, 698, 17)
print(f"Correlaciones entre customer 698 y 17:")
print(corr_698_17)

Correlaciones entre customer 698 y 17:
(698, 2232)  (17, 2379)    1.000000
(698, 2069)  (17, 1242)    1.000000
(698, 2136)  (17, 1711)    0.880705
(698, 1866)  (17, 1711)    0.320256
(698, 2066)  (17, 1242)   -0.166667
             (17, 2379)   -0.166667
(698, 1082)  (17, 1242)   -0.166667
(698, 1924)  (17, 1242)   -0.166667
             (17, 2379)   -0.166667
(698, 1866)  (17, 2379)   -0.166667
             (17, 1242)   -0.166667
(698, 2069)  (17, 2379)   -0.166667
(698, 2136)  (17, 1242)   -0.166667
(698, 1082)  (17, 2379)   -0.166667
(698, 2136)  (17, 2379)   -0.166667
(698, 2232)  (17, 1242)   -0.166667
(698, 1082)  (17, 1711)   -0.240192
(698, 1924)  (17, 1711)   -0.240192
(698, 2069)  (17, 1711)   -0.240192
(698, 2232)  (17, 1711)   -0.240192
(698, 2066)  (17, 1711)   -0.240192
dtype: float64


In [112]:
def get_average_correlation_between_customers(corr_series, customer_id_1, customer_id_2):
    """
    Calcula el promedio de correlaciones entre dos customers específicos
    """
    # Obtener todas las correlaciones entre los dos customers
    correlations = get_correlation_between_customers(corr_series, customer_id_1, customer_id_2)
    
    if len(correlations) == 0:
        print(f"No se encontraron correlaciones entre customer {customer_id_1} y {customer_id_2}")
        return None
    
    # Calcular el promedio
    average_corr = correlations.mean()
    
    print(f"Promedio de correlación entre customer {customer_id_1} y {customer_id_2}: {average_corr:.4f}")
    print(f"Basado en {len(correlations)} pares de órdenes")
    
    return average_corr

# Ejemplo
avg_corr_698_17 = get_average_correlation_between_customers(correlaciones_serie, 698, 17)

Promedio de correlación entre customer 698 y 17: -0.0000
Basado en 21 pares de órdenes


In [113]:
def get_unique_customers(corr_series):
    """
    Obtiene todos los customer_ids únicos del dataset
    """
    # Extraer customer_ids de ambos niveles del multi-index
    customers_level_0 = corr_series.index.get_level_values(0).str[0].unique()
    customers_level_1 = corr_series.index.get_level_values(1).str[0].unique()
    
    # Combinar y obtener valores únicos
    all_customers = sorted(set(customers_level_0) | set(customers_level_1))
    
    print(f"Se encontraron {len(all_customers)} customers únicos")
    return all_customers

# Obtener todos los customers
unique_customers = get_unique_customers(correlaciones_serie)
print("Customers únicos:", unique_customers[:10], "...")  # Mostrar primeros 10

Se encontraron 568 customers únicos
Customers únicos: [1, 2, 3, 4, 5, 7, 8, 9, 10, 11] ...


In [115]:
correlaciones_serie

(1, 2460)    (1, 1990)     -0.254588
(2, 2444)    (1, 1990)     -0.222222
             (1, 2460)      0.424313
(3, 1197)    (1, 1990)     -0.166667
             (1, 2460)      0.509175
                              ...   
(798, 1286)  (793, 1064)   -0.166667
             (794, 1840)   -0.166667
             (796, 1480)   -0.166667
             (796, 1837)   -0.166667
             (796, 2491)    1.000000
Length: 494515, dtype: float64

In [119]:
def count_perfect_correlations_by_customer_pair(corr_series):
    """
    Cuenta cuántos pares de órdenes tienen correlación 1.0 por cada combinación de customers
    """
    from collections import defaultdict
    
    # Diccionario para contar: (customer1, customer2) -> count
    perfect_corr_count = defaultdict(int)
    
    # Filtrar solo correlaciones perfectas (1.0)
    perfect_correlations = corr_series[corr_series == 1.0]
    
    for (pair1, pair2), correlation in perfect_correlations.items():
        cust1, order1 = pair1
        cust2, order2 = pair2
        
        # Crear clave ordenada para evitar duplicados (cust1, cust2) y (cust2, cust1)
        key = tuple(sorted([cust1, cust2]))
        perfect_corr_count[key] += 1
    
    return perfect_corr_count

# Contar pares con correlación perfecta
perfect_counts = count_perfect_correlations_by_customer_pair(correlaciones_serie)

# Mostrar resultados
print("Pares de customers con correlaciones perfectas (1.0) y su conteo:")
for (cust1, cust2), count in sorted(perfect_counts.items(), key=lambda x: x[1], reverse=True):
    if count > 4:
        print(f"({cust1}, {cust2}): {count} pares con correlación 1.0")

Pares de customers con correlaciones perfectas (1.0) y su conteo:
(45, 180): 7 pares con correlación 1.0
(180, 500): 7 pares con correlación 1.0
(45, 547): 7 pares con correlación 1.0
(45, 667): 7 pares con correlación 1.0
(180, 667): 7 pares con correlación 1.0
(547, 698): 7 pares con correlación 1.0
(45, 717): 7 pares con correlación 1.0
(180, 717): 7 pares con correlación 1.0
(45, 755): 7 pares con correlación 1.0
(45, 240): 6 pares con correlación 1.0
(45, 500): 6 pares con correlación 1.0
(547, 667): 6 pares con correlación 1.0
(500, 698): 6 pares con correlación 1.0
(667, 698): 6 pares con correlación 1.0
(547, 717): 6 pares con correlación 1.0
(667, 717): 6 pares con correlación 1.0
(180, 755): 6 pares con correlación 1.0
(240, 755): 6 pares con correlación 1.0
(500, 755): 6 pares con correlación 1.0
(30, 45): 5 pares con correlación 1.0
(30, 180): 5 pares con correlación 1.0
(45, 228): 5 pares con correlación 1.0
(180, 228): 5 pares con correlación 1.0
(180, 240): 5 pares con c

In [120]:
def find_customer_clusters(perfect_counts, min_connections=3):
    """
    Encuentra clusters de customers que están todos conectados entre sí
    """
    from collections import defaultdict
    
    # Crear grafo de conexiones
    graph = defaultdict(set)
    for (cust1, cust2), count in perfect_counts.items():
        if count >= min_connections:
            graph[cust1].add(cust2)
            graph[cust2].add(cust1)
    
    # Encontrar componentes conectados (clusters)
    visited = set()
    clusters = []
    
    def dfs(customer, cluster):
        visited.add(customer)
        cluster.add(customer)
        for neighbor in graph[customer]:
            if neighbor not in visited:
                dfs(neighbor, cluster)
    
    for customer in graph:
        if customer not in visited:
            cluster = set()
            dfs(customer, cluster)
            if len(cluster) > 1:  # Solo clusters con múltiples customers
                clusters.append(cluster)
    
    return clusters

# Encontrar clusters
clusters = find_customer_clusters(perfect_counts, min_connections=5)
print("Clusters de customers con comportamientos idénticos:")
for i, cluster in enumerate(clusters, 1):
    print(f"Cluster {i}: {sorted(cluster)}")

Clusters de customers con comportamientos idénticos:
Cluster 1: [5, 30, 45, 180, 228, 240, 405, 463, 500, 543, 547, 620, 667, 698, 717, 730, 755]


In [126]:
pivot_df2.loc[180]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1019,0.0,0.0,0.0,0.0,0.0,42.0,0.0
1078,0.0,0.0,0.0,0.0,0.0,0.0,21.0
1094,0.0,0.0,45.0,0.0,0.0,0.0,0.0
1364,4.0,0.0,0.0,0.0,0.0,0.0,0.0
1545,0.0,0.0,0.0,28.0,0.0,0.0,0.0
2020,0.0,14.0,0.0,0.0,0.0,0.0,0.0
2370,0.0,0.0,0.0,0.0,30.0,0.0,0.0


In [122]:
def analyze_customer_45_cluster(perfect_counts):
    """
    Analiza específicamente el cluster alrededor del customer 45
    """
    # Customers que tienen alta correlación con 45
    customers_connected_to_45 = []
    for (cust1, cust2), count in perfect_counts.items():
        if count >= 5:  # Umbral alto
            if cust1 == 45:
                customers_connected_to_45.append((cust2, count))
            elif cust2 == 45:
                customers_connected_to_45.append((cust1, count))
    
    # Ordenar por número de pares
    customers_connected_to_45.sort(key=lambda x: x[1], reverse=True)
    
    print("Customers fuertemente conectados con 45:")
    for cust, count in customers_connected_to_45:
        print(f"  Customer {cust}: {count} pares perfectos")
    
    return customers_connected_to_45

# Analizar conexiones del 45
connections_45 = analyze_customer_45_cluster(perfect_counts)

Customers fuertemente conectados con 45:
  Customer 180: 7 pares perfectos
  Customer 547: 7 pares perfectos
  Customer 667: 7 pares perfectos
  Customer 717: 7 pares perfectos
  Customer 755: 7 pares perfectos
  Customer 240: 6 pares perfectos
  Customer 500: 6 pares perfectos
  Customer 30: 5 pares perfectos
  Customer 228: 5 pares perfectos
  Customer 463: 5 pares perfectos
  Customer 698: 5 pares perfectos


In [138]:
pivot_df2.loc[45]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1160,0.0,0.0,0.0,0.0,15.0,0.0,0.0
1388,0.0,14.0,0.0,0.0,0.0,0.0,0.0
1490,0.0,0.0,0.0,36.0,0.0,0.0,0.0
1871,0.0,0.0,21.0,0.0,0.0,0.0,0.0
1895,0.0,0.0,0.0,0.0,0.0,0.0,14.0
1952,0.0,0.0,0.0,0.0,0.0,66.0,0.0
1957,6.0,0.0,0.0,0.0,0.0,0.0,0.0


In [139]:
pivot_df2.loc[180]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1019,0.0,0.0,0.0,0.0,0.0,42.0,0.0
1078,0.0,0.0,0.0,0.0,0.0,0.0,21.0
1094,0.0,0.0,45.0,0.0,0.0,0.0,0.0
1364,4.0,0.0,0.0,0.0,0.0,0.0,0.0
1545,0.0,0.0,0.0,28.0,0.0,0.0,0.0
2020,0.0,14.0,0.0,0.0,0.0,0.0,0.0
2370,0.0,0.0,0.0,0.0,30.0,0.0,0.0


In [128]:
pivot_df2.loc[547]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1040,8.0,0.0,0.0,0.0,0.0,0.0,0.0
1332,0.0,0.0,0.0,0.0,25.0,0.0,0.0
1538,0.0,0.0,0.0,0.0,0.0,0.0,21.0
1783,0.0,0.0,0.0,20.0,0.0,0.0,0.0
1880,0.0,20.0,0.0,0.0,0.0,0.0,0.0
1890,0.0,0.0,24.0,0.0,0.0,0.0,0.0
2016,0.0,0.0,0.0,0.0,0.0,60.0,0.0


In [129]:
pivot_df2.loc[667]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1395,0.0,0.0,0.0,0.0,25.0,0.0,0.0
1396,0.0,0.0,0.0,0.0,0.0,30.0,0.0
1520,3.0,0.0,0.0,0.0,0.0,0.0,0.0
1759,0.0,0.0,30.0,0.0,0.0,0.0,0.0
1971,0.0,0.0,0.0,0.0,0.0,0.0,35.0
2308,0.0,10.0,0.0,0.0,0.0,0.0,0.0
2410,0.0,0.0,0.0,32.0,0.0,0.0,0.0


In [130]:
pivot_df2.loc[717]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1182,0.0,0.0,0.0,0.0,45.0,0.0,0.0
1397,0.0,10.0,0.0,0.0,0.0,0.0,0.0
1443,0.0,0.0,0.0,0.0,0.0,78.0,0.0
1563,6.0,0.0,0.0,0.0,0.0,0.0,0.0
1787,0.0,0.0,0.0,0.0,0.0,0.0,35.0
2048,0.0,0.0,36.0,0.0,0.0,0.0,0.0
2304,0.0,0.0,0.0,4.0,0.0,0.0,0.0


In [131]:
pivot_df2.loc[755]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1085,0.0,0.0,0.0,32.0,0.0,0.0,0.0
1164,0.0,0.0,0.0,0.0,0.0,0.0,21.0
1184,0.0,0.0,27.0,0.0,0.0,0.0,0.0
1552,0.0,22.0,0.0,0.0,0.0,0.0,0.0
2083,4.0,0.0,0.0,0.0,0.0,0.0,0.0
2103,0.0,0.0,0.0,0.0,0.0,36.0,0.0
2210,0.0,0.0,0.0,0.0,15.0,0.0,0.0


In [132]:
pivot_df2.loc[240]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1027,0.0,0.0,0.0,36.0,0.0,0.0,0.0
1067,5.0,0.0,0.0,0.0,0.0,0.0,0.0
1168,0.0,0.0,0.0,0.0,0.0,66.0,0.0
1343,0.0,6.0,0.0,0.0,0.0,0.0,0.0
1417,0.0,0.0,0.0,0.0,0.0,0.0,35.0
2119,0.0,0.0,24.0,0.0,0.0,0.0,0.0
2456,0.0,0.0,0.0,0.0,30.0,0.0,0.0


In [133]:
pivot_df2.loc[500]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1384,0.0,0.0,0.0,0.0,0.0,0.0,42.0
1394,0.0,0.0,42.0,0.0,0.0,0.0,0.0
1709,0.0,0.0,0.0,0.0,0.0,54.0,0.0
1948,6.0,0.0,0.0,0.0,0.0,0.0,0.0
2231,0.0,0.0,0.0,24.0,0.0,0.0,0.0
2481,0.0,12.0,0.0,0.0,0.0,0.0,0.0
2493,0.0,0.0,0.0,0.0,35.0,0.0,0.0


In [134]:
pivot_df2.loc[30]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1146,0.0,0.0,0.0,28.0,0.0,0.0,0.0
1219,0.0,22.0,0.0,0.0,0.0,0.0,0.0
1466,0.0,0.0,24.0,0.0,20.0,0.0,0.0
1484,2.0,0.0,0.0,0.0,0.0,0.0,0.0
1733,0.0,0.0,0.0,0.0,0.0,0.0,35.0
2110,0.0,0.0,0.0,0.0,0.0,90.0,0.0


In [135]:
pivot_df2.loc[228]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1401,0.0,0.0,24.0,0.0,35.0,0.0,0.0
1478,0.0,22.0,0.0,0.0,0.0,0.0,0.0
1722,0.0,0.0,0.0,0.0,0.0,0.0,42.0
1776,0.0,0.0,0.0,0.0,0.0,24.0,0.0
1811,0.0,0.0,0.0,16.0,0.0,0.0,0.0
1918,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [136]:
pivot_df2.loc[463]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1248,3.0,0.0,0.0,0.0,0.0,0.0,35.0
1319,0.0,20.0,0.0,0.0,0.0,0.0,0.0
1450,0.0,0.0,18.0,0.0,0.0,0.0,0.0
1881,0.0,0.0,0.0,0.0,0.0,48.0,0.0
2013,0.0,0.0,0.0,0.0,15.0,0.0,0.0
2107,0.0,0.0,0.0,16.0,0.0,0.0,0.0


In [137]:
pivot_df2.loc[698]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1082,3.0,0.0,0.0,0.0,0.0,0.0,0.0
1866,0.0,18.0,0.0,0.0,0.0,0.0,0.0
1924,0.0,0.0,0.0,0.0,0.0,0.0,7.0
2066,0.0,0.0,0.0,0.0,0.0,54.0,0.0
2069,0.0,0.0,15.0,0.0,0.0,0.0,0.0
2136,0.0,0.0,0.0,20.0,0.0,0.0,0.0
2232,0.0,0.0,0.0,0.0,15.0,0.0,0.0


In [143]:
# Filas en posiciones 0, 5 y 10
df_reducido = pivot_df2.loc[[180, 547, 667, 717, 755, 240, 500, 30, 228, 463, 698]]


In [144]:
df_reducido

pizza_id                1    2     3     4     5     6     7
customer_id order_id                                        
180         1019      0.0  0.0   0.0   0.0   0.0  42.0   0.0
            1078      0.0  0.0   0.0   0.0   0.0   0.0  21.0
            1094      0.0  0.0  45.0   0.0   0.0   0.0   0.0
            1364      4.0  0.0   0.0   0.0   0.0   0.0   0.0
            1545      0.0  0.0   0.0  28.0   0.0   0.0   0.0
...                   ...  ...   ...   ...   ...   ...   ...
698         1924      0.0  0.0   0.0   0.0   0.0   0.0   7.0
            2066      0.0  0.0   0.0   0.0   0.0  54.0   0.0
            2069      0.0  0.0  15.0   0.0   0.0   0.0   0.0
            2136      0.0  0.0   0.0  20.0   0.0   0.0   0.0
            2232      0.0  0.0   0.0   0.0  15.0   0.0   0.0

[74 rows x 7 columns]

In [147]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# df_pedidos es tu DataFrame completo de pedidos
sim_matrix = cosine_similarity(df_reducido)

# Convertir a DataFrame para mayor claridad
sim_df = pd.DataFrame(sim_matrix, index=df_reducido.index, columns=df_reducido.index)

# Ver los pedidos más similares a un pedido específico
pedido_ejemplo = df_reducido.index[0]
sim_df[pedido_ejemplo].sort_values(ascending=False)


customer_id  order_id
180          1019        1.0
547          2016        1.0
698          2066        1.0
463          1881        1.0
228          1776        1.0
                        ... 
180          1078        0.0
240          1343        0.0
             1417        0.0
             2119        0.0
698          2232        0.0
Name: (180, 1019), Length: 74, dtype: float64

In [149]:
import numpy as np

# Ignorar la diagonal (similitud de un pedido consigo mismo)
np.fill_diagonal(sim_matrix, 0)

# Índices del par de pedidos más parecidos
max_sim_idx = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
pedido1, pedido2 = df_reducido.index[max_sim_idx[0]], df_reducido.index[max_sim_idx[1]]
print("Pedidos más parecidos:", pedido1, pedido2)
print("Similitud:", sim_matrix[max_sim_idx])


Pedidos más parecidos: (180, 1019) (547, 2016)
Similitud: 1.0


In [151]:
pivot_df2.loc[180]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1019,0.0,0.0,0.0,0.0,0.0,42.0,0.0
1078,0.0,0.0,0.0,0.0,0.0,0.0,21.0
1094,0.0,0.0,45.0,0.0,0.0,0.0,0.0
1364,4.0,0.0,0.0,0.0,0.0,0.0,0.0
1545,0.0,0.0,0.0,28.0,0.0,0.0,0.0
2020,0.0,14.0,0.0,0.0,0.0,0.0,0.0
2370,0.0,0.0,0.0,0.0,30.0,0.0,0.0


In [152]:
pivot_df2.loc[547]

pizza_id,1,2,3,4,5,6,7
order_id,,,,,,,
1040,8.0,0.0,0.0,0.0,0.0,0.0,0.0
1332,0.0,0.0,0.0,0.0,25.0,0.0,0.0
1538,0.0,0.0,0.0,0.0,0.0,0.0,21.0
1783,0.0,0.0,0.0,20.0,0.0,0.0,0.0
1880,0.0,20.0,0.0,0.0,0.0,0.0,0.0
1890,0.0,0.0,24.0,0.0,0.0,0.0,0.0
2016,0.0,0.0,0.0,0.0,0.0,60.0,0.0


In [155]:
import numpy as np

# Ignorar la diagonal (similitud de un pedido consigo mismo)
np.fill_diagonal(sim_matrix, 0)

# Índices del par de pedidos más parecidos
max_sim_idx = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
pedido1, pedido2 = pivot_df2.index[max_sim_idx[0]], pivot_df2.index[max_sim_idx[1]]
print("Pedidos más parecidos:", pedido1, pedido2)
print("Similitud:", sim_matrix[max_sim_idx])


Pedidos más parecidos: (1, 1990) (8, 1326)
Similitud: 1.0


In [157]:
# Convertir la matriz de similitud en una lista de pares
pairs = []

for i, idx1 in enumerate(pivot_df2.index):
    for j, idx2 in enumerate(pivot_df2.index):
        if i < j:  # evitar duplicados y la diagonal
            pairs.append((idx1, idx2, sim_matrix[i, j]))

# Convertir a DataFrame
pairs_df = pd.DataFrame(pairs, columns=['pedido_1', 'pedido_2', 'similitud'])


IndexError: index 74 is out of bounds for axis 1 with size 74